In [1]:
import os, sys, importlib
import duckdb
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
pd.set_option('display.max_columns', None)
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)
import io_utils.db_saver
importlib.reload(io_utils.db_saver)
from io_utils.db_saver import register_or_update_job, save_e2_to_db, save_tripinfo_to_db, save_shp_to_4326db, inspect_database
import io_utils.db_reader
importlib.reload(io_utils.db_reader)
from io_utils.db_reader import load_table_to_df

In [5]:
db_path = os.path.join("..", "sumo_output.db")
con = duckdb.connect(db_path)
# my_job = "20260127"
my_job = "20260225"

In [3]:
inspect_database(db_path)

📂 Database File: c:\Users\mmh\Documents\sumo_visualization\sumo_output.db
⚖️  File Size: 180.76 MB

📊 Database Structure:
--------------------------------------------------
Table: **all_e2**
  └─ Rows: 4,552
  └─ Columns: begin, end, id, sampledSeconds, nVehEntered, nVehLeft, nVehSeen, meanSpeed, meanTimeLoss, meanOccupancy, maxOccupancy, meanMaxJamLengthInVehicles, meanMaxJamLengthInMeters, maxJamLengthInVehicles, maxJamLengthInMeters, jamLengthInVehiclesSum, jamLengthInMetersSum, meanHaltingDuration, maxHaltingDuration, haltingDurationSum, meanIntervalHaltingDuration, maxIntervalHaltingDuration, intervalHaltingDurationSum, startedHalts, meanVehicleNumber, maxVehicleNumber, sim_job_id
--------------------------------------------------
Table: **all_trips**
  └─ Rows: 13,398
  └─ Columns: id, depart, departLane, departPos, departSpeed, departDelay, arrival, arrivalLane, arrivalPos, arrivalSpeed, duration, routeLength, waitingTime, waitingCount, stopTime, timeLoss, rerouteNo, devices, vT

# Saving

In [4]:
register_or_update_job(my_job, "charging", "orig network add charging", con)

📝 Registered new job: 20260225


In [6]:
model_dir = r"C:\Users\mmh\OneDrive - Oak Ridge National Laboratory\nashville_microsim\Model\SUMO\xml"
filename_e2 = os.path.join(model_dir,"model baseline","e2_output.xml")
filename_trip = os.path.join(model_dir,"model baseline","tripinfo.xml")
save_e2_to_db(filename_e2, my_job, con, rewrite=True)
save_tripinfo_to_db(filename_trip, my_job, con, rewrite=True)

✅ Job '20260225' data rewritten in all_e2 table.
✅ TripInfo for Job '20260225' rewritten in table all_trips.


In [17]:
model_dir = r"C:\Users\mmh\OneDrive - Oak Ridge National Laboratory\nashville_microsim\Model\SUMO\shp"
filename_edge = os.path.join(model_dir,"links_sumo_wcc.shp")
filename_node = os.path.join(model_dir,"nodes_sumo_wcc.shp")
save_shp_to_4326db(filename_edge, my_job, "edge", con)
save_shp_to_4326db(filename_node, my_job, "node", con)

✅ Geometry table 'edge' created/overwritten.
✅ Geometry table 'node' created/overwritten.


In [7]:
con.close()